# Gold Business Aggregates

This notebook builds dashboard-ready aggregate models from the Gold dimensions and facts.

The aggregates support sales, customer geography, product, seller, and delivery analysis.

In [0]:
from pyspark.sql import functions as F

## 1. Define Gold storage paths

In [0]:
GOLD_BASE_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist"
)

DIM_CUSTOMERS_PATH = f"{GOLD_BASE_PATH}/dim_customers"
DIM_PRODUCTS_PATH = f"{GOLD_BASE_PATH}/dim_products"
DIM_SELLERS_PATH = f"{GOLD_BASE_PATH}/dim_sellers"
DIM_DATES_PATH = f"{GOLD_BASE_PATH}/dim_dates"

FACT_ORDERS_PATH = f"{GOLD_BASE_PATH}/fact_orders"
FACT_ORDER_ITEMS_PATH = f"{GOLD_BASE_PATH}/fact_order_items"
FACT_PAYMENTS_PATH = f"{GOLD_BASE_PATH}/fact_payments"

SALES_DAILY_PATH = f"{GOLD_BASE_PATH}/sales_daily"
SALES_BY_STATE_PATH = f"{GOLD_BASE_PATH}/sales_by_state"
PRODUCT_PERFORMANCE_PATH = f"{GOLD_BASE_PATH}/product_performance"
SELLER_PERFORMANCE_PATH = f"{GOLD_BASE_PATH}/seller_performance"
DELIVERY_PERFORMANCE_PATH = f"{GOLD_BASE_PATH}/delivery_performance"

## 2. Read Gold inputs

In [0]:
dim_customers_df = spark.read.format("delta").load(DIM_CUSTOMERS_PATH)
dim_products_df = spark.read.format("delta").load(DIM_PRODUCTS_PATH)
dim_sellers_df = spark.read.format("delta").load(DIM_SELLERS_PATH)
dim_dates_df = spark.read.format("delta").load(DIM_DATES_PATH)

fact_orders_df = spark.read.format("delta").load(FACT_ORDERS_PATH)
fact_order_items_df = spark.read.format("delta").load(FACT_ORDER_ITEMS_PATH)
fact_payments_df = spark.read.format("delta").load(FACT_PAYMENTS_PATH)

print(f"Customers: {dim_customers_df.count():,}")
print(f"Products: {dim_products_df.count():,}")
print(f"Sellers: {dim_sellers_df.count():,}")
print(f"Dates: {dim_dates_df.count():,}")
print(f"Orders: {fact_orders_df.count():,}")
print(f"Order items: {fact_order_items_df.count():,}")
print(f"Payments: {fact_payments_df.count():,}")

## 3. Build order-level financial metrics

In [0]:
order_item_totals_df = (
    fact_order_items_df
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),
        F.sum("price").alias("product_value"),
        F.sum("freight_value").alias("freight_value"),
        F.sum("item_total_value").alias("item_total_value"),
    )
)

order_payment_totals_df = (
    fact_payments_df
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_value"),
        F.count("*").alias("payment_count"),
    )
)

order_metrics_df = (
    fact_orders_df.alias("orders")
    .join(
        order_item_totals_df.alias("items"),
        on="order_id",
        how="left",
    )
    .join(
        order_payment_totals_df.alias("payments"),
        on="order_id",
        how="left",
    )
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "purchase_date_key",
        "delivery_days",
        "delivery_delay_days",
        "is_delayed",
        F.coalesce(F.col("item_count"), F.lit(0)).alias("item_count"),
        F.coalesce(F.col("product_value"), F.lit(0.0)).alias("product_value"),
        F.coalesce(F.col("freight_value"), F.lit(0.0)).alias("freight_value"),
        F.coalesce(F.col("item_total_value"), F.lit(0.0)).alias("item_total_value"),
        F.coalesce(F.col("payment_value"), F.lit(0.0)).alias("payment_value"),
        F.coalesce(F.col("payment_count"), F.lit(0)).alias("payment_count"),
    )
)

display(order_metrics_df.limit(10))

## 4. Build daily sales aggregate

In [0]:
sales_daily_df = (
    order_metrics_df.alias("orders")
    .join(
        dim_dates_df.alias("dates"),
        F.col("orders.purchase_date_key") == F.col("dates.date_key"),
        how="inner",
    )
    .groupBy(
        F.col("dates.date_key"),
        F.col("dates.date"),
        F.col("dates.year"),
        F.col("dates.quarter"),
        F.col("dates.month"),
        F.col("dates.month_name"),
        F.col("dates.day_name"),
        F.col("dates.is_weekend"),
    )
    .agg(
        F.countDistinct("orders.order_id").alias("order_count"),
        F.countDistinct("orders.customer_id").alias("customer_count"),
        F.sum("orders.item_count").alias("item_count"),
        F.sum("orders.payment_value").alias("revenue"),
        F.sum("orders.product_value").alias("product_value"),
        F.sum("orders.freight_value").alias("freight_value"),
        F.avg("orders.payment_value").alias("average_order_value"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(sales_daily_df.orderBy("date").limit(10))

## 5. Build sales by customer state

In [0]:
sales_by_state_df = (
    order_metrics_df.alias("orders")
    .join(
        dim_customers_df.alias("customers"),
        on="customer_id",
        how="inner",
    )
    .groupBy(
        "customers.customer_state",
        "customers.customer_region",
    )
    .agg(
        F.countDistinct("orders.order_id").alias("order_count"),
        F.countDistinct("orders.customer_id").alias("customer_count"),
        F.sum("orders.item_count").alias("item_count"),
        F.sum("orders.payment_value").alias("revenue"),
        F.avg("orders.payment_value").alias("average_order_value"),
        F.avg("orders.delivery_days").alias("average_delivery_days"),
        F.sum(
            F.when(F.col("orders.is_delayed") == True, 1).otherwise(0)
        ).alias("delayed_order_count"),
    )
    .withColumn(
        "delayed_order_rate",
        F.when(
            F.col("order_count") > 0,
            F.col("delayed_order_count") / F.col("order_count"),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(
    sales_by_state_df
    .orderBy(F.desc("revenue"))
)

## 6. Build product performance aggregate

In [0]:
product_performance_df = (
    fact_order_items_df.alias("items")
    .join(
        dim_products_df.alias("products"),
        on="product_id",
        how="inner",
    )
    .groupBy(
        "items.product_id",
        "products.product_category_name",
        "products.product_category_name_english",
    )
    .agg(
        F.count("*").alias("units_sold"),
        F.countDistinct("items.order_id").alias("order_count"),
        F.countDistinct("items.seller_id").alias("seller_count"),
        F.sum("items.price").alias("product_revenue"),
        F.sum("items.freight_value").alias("freight_value"),
        F.sum("items.item_total_value").alias("total_value"),
        F.avg("items.price").alias("average_unit_price"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(
    product_performance_df
    .orderBy(F.desc("product_revenue"))
    .limit(20)
)

## 7. Build seller performance aggregate

In [0]:
seller_performance_df = (
    fact_order_items_df.alias("items")
    .join(
        dim_sellers_df.alias("sellers"),
        on="seller_id",
        how="inner",
    )
    .groupBy(
        "items.seller_id",
        "sellers.seller_city",
        "sellers.seller_state",
        "sellers.seller_region",
    )
    .agg(
        F.countDistinct("items.order_id").alias("order_count"),
        F.count("*").alias("units_sold"),
        F.countDistinct("items.product_id").alias("product_count"),
        F.sum("items.price").alias("product_revenue"),
        F.sum("items.freight_value").alias("freight_value"),
        F.sum("items.item_total_value").alias("total_value"),
        F.avg("items.price").alias("average_unit_price"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(
    seller_performance_df
    .orderBy(F.desc("product_revenue"))
    .limit(20)
)

## 8. Build delivery performance aggregate

In [0]:
delivery_performance_df = (
    order_metrics_df
    .filter(F.col("order_status") == "delivered")
    .groupBy("purchase_date_key")
    .agg(
        F.countDistinct("order_id").alias("delivered_order_count"),
        F.avg("delivery_days").alias("average_delivery_days"),
        F.avg("delivery_delay_days").alias("average_delivery_delay_days"),
        F.sum(
            F.when(F.col("is_delayed") == True, 1).otherwise(0)
        ).alias("delayed_order_count"),
        F.sum(
            F.when(F.col("is_delayed") == False, 1).otherwise(0)
        ).alias("on_time_order_count"),
    )
    .withColumn(
        "delayed_order_rate",
        F.when(
            F.col("delivered_order_count") > 0,
            F.col("delayed_order_count")
            / F.col("delivered_order_count"),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(
    delivery_performance_df
    .orderBy("purchase_date_key")
    .limit(20)
)

## 9. Validate business aggregates

In [0]:
aggregate_checks = [
    (
        "sales_daily",
        sales_daily_df,
        ["date_key"],
    ),
    (
        "sales_by_state",
        sales_by_state_df,
        ["customer_state"],
    ),
    (
        "product_performance",
        product_performance_df,
        ["product_id"],
    ),
    (
        "seller_performance",
        seller_performance_df,
        ["seller_id"],
    ),
    (
        "delivery_performance",
        delivery_performance_df,
        ["purchase_date_key"],
    ),
]

for aggregate_name, aggregate_df, grain_columns in aggregate_checks:
    row_count = aggregate_df.count()

    duplicate_count = (
        aggregate_df
        .groupBy(*grain_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if row_count == 0:
        raise ValueError(f"{aggregate_name} is empty.")

    if duplicate_count > 0:
        raise ValueError(
            f"{aggregate_name} contains "
            f"{duplicate_count:,} duplicate grain combinations."
        )

    print(
        f"{aggregate_name}: {row_count:,} rows — validation passed."
    )

## 10. Write business aggregates to Gold

In [0]:
aggregate_outputs = [
    (sales_daily_df, SALES_DAILY_PATH),
    (sales_by_state_df, SALES_BY_STATE_PATH),
    (product_performance_df, PRODUCT_PERFORMANCE_PATH),
    (seller_performance_df, SELLER_PERFORMANCE_PATH),
    (delivery_performance_df, DELIVERY_PERFORMANCE_PATH),
]

for aggregate_df, output_path in aggregate_outputs:
    (
        aggregate_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(output_path)
    )

    print(f"Written: {output_path}")

## 11. Validate Gold outputs

In [0]:
aggregate_readback_checks = [
    ("sales_daily", sales_daily_df, SALES_DAILY_PATH),
    ("sales_by_state", sales_by_state_df, SALES_BY_STATE_PATH),
    (
        "product_performance",
        product_performance_df,
        PRODUCT_PERFORMANCE_PATH,
    ),
    (
        "seller_performance",
        seller_performance_df,
        SELLER_PERFORMANCE_PATH,
    ),
    (
        "delivery_performance",
        delivery_performance_df,
        DELIVERY_PERFORMANCE_PATH,
    ),
]

for aggregate_name, expected_df, output_path in aggregate_readback_checks:
    expected_count = expected_df.count()

    written_df = (
        spark.read
        .format("delta")
        .load(output_path)
    )

    written_count = written_df.count()

    if written_count != expected_count:
        raise ValueError(
            f"{aggregate_name} write validation failed. "
            f"Expected: {expected_count:,}, "
            f"Written: {written_count:,}"
        )

    print(
        f"{aggregate_name}: {written_count:,} written rows — validation passed."
    )